# Qwen2.5-VL-3B PRM Generation (Colab / Kaggle)
This notebook clones our repository, installs the required dependencies, downloads the CharXiv dataset via HuggingFace (including images), and runs the step-by-step reasoning generation for a subset of samples.

Results are incrementally saved to a `.jsonl` file to prevent data loss.

In [ ]:
!git clone https://github.com/yahorlahunovich/prm_project.git
%cd prm_project
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils pillow torchvision

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import json
import os

# Load Model in 4-bit precision to fit within the 16GB VRAM of a T4 GPU
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
from datasets import load_dataset
import sys
sys.path.append('src')
from chart_prm.generator import build_generation_prompt

# Download the dataset directly through HuggingFace (images will be cached locally)
print("Downloading CharXiv...")
dataset = load_dataset("princeton-nlp/CharXiv", split="descriptive_val")

# Limit to the first 100 samples for initial testing
num_samples = 100
dataset = dataset.select(range(num_samples))

In [ ]:
output_file = "generated_reasoning_steps.jsonl"
save_every = 10
results = []

for i, sample in enumerate(dataset):
    image = sample['image'] # This is automatically loaded as a PIL Image by HF
    question = sample['question']
    
    prompt_text = build_generation_prompt(question)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]
    
    # Formatting for Qwen2.5-VL
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    
    # Autoregressive generation
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)
        
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    
    # Prepare result row
    res = {
        "sample_index": i,
        "question": question,
        "model_output": output_text
    }
    # Attempt to grab answer or id if available in the dataset schema
    if 'answer' in sample: res['ground_truth'] = sample['answer']
    if 'question_id' in sample: res['question_id'] = sample['question_id']
    
    results.append(res)
    
    # Save checkpoints periodically
    if (i + 1) % save_every == 0 or (i + 1) == num_samples:
        print(f"Processed {i+1}/{num_samples}. Saving checkpoint...")
        with open(output_file, "w", encoding="utf-8") as f:
            for r in results:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Finished generating {num_samples} trajectories!")
